# MOOCCubeX — Build dimensions và bridges trên Kaggle CPU

Notebook này đọc tầng Parquet do `01_moocubex_raw_to_parquet.ipynb` tạo ra và xây tầng catalog chuẩn hóa trước khi bung event lớn.

Đầu ra chính:

- `dim_course`, `dim_problem`, `dim_video`, `dim_resource`;
- `dim_lo_resource`, `dim_lo_catalog`;
- `bridge_video_ccid`, `bridge_exercise_problem`, `bridge_lo_problem`;
- `bridge_user_course_enrollment`;
- các bridge comment nếu nguồn tương ứng tồn tại;
- quarantine, join coverage, manifest và `_SUCCESS`.

LO được định nghĩa là **big chapter**: `1.1`, `1.1.1`, `1.2` đều thành chapter major `1`. Notebook chỉ dùng CPU, DuckDB out-of-core và ghi checkpoint sau từng bảng.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

try:
    import duckdb
except ModuleNotFoundError:
    print("DuckDB chưa có; đang cài duckdb>=1.3 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.3"])
    import duckdb

# Nếu output notebook 01 đã được publish thành Kaggle Dataset, có thể để None
# để notebook tự tìm catalog.json. Chỉ đặt path khi có nhiều dataset phù hợp.
PARQUET_ROOT: Optional[str] = None
OUTPUT_ROOT = Path("/kaggle/working/moocubex_model")

FORCE_REBUILD = False
THREADS = min(os.cpu_count() or 4, 8)
MEMORY_FRACTION = 0.72
ROW_GROUP_SIZE = 250_000
ZSTD_LEVEL = 3

print("Python:", sys.version.split()[0])
print("DuckDB:", duckdb.__version__)
print("THREADS:", THREADS)


## 1. Tự tìm output của notebook 01

Không sử dụng các absolute path cũ ghi trong `catalog.json`, vì khi publish output thành Kaggle Dataset đường dẫn `/kaggle/working/...` sẽ đổi thành `/kaggle/input/...`. Thay vào đó notebook dựng lại path từ layout chuẩn `tables/<group>/<table>/*.parquet`.


In [ ]:
def is_parquet_root(path: Path) -> bool:
    return path.is_dir() and (path / "tables").is_dir() and (path / "catalog.json").is_file()


def discover_parquet_root(configured: Optional[str]) -> Path:
    if configured:
        root = Path(configured).resolve()
        if not is_parquet_root(root):
            raise FileNotFoundError(
                f"PARQUET_ROOT không đúng layout notebook 01: {root}. "
                "Cần có catalog.json và thư mục tables/."
            )
        return root

    candidates: List[Path] = []
    direct = Path("/kaggle/working/moocubex_parquet")
    if is_parquet_root(direct):
        candidates.append(direct.resolve())

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for catalog_path in kaggle_input.rglob("catalog.json"):
            root = catalog_path.parent.resolve()
            if is_parquet_root(root) and root not in candidates:
                candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy output của notebook 01. Hãy attach Kaggle Dataset chứa "
            "moocubex_parquet/catalog.json và tables/, hoặc đặt PARQUET_ROOT."
        )

    print("Các Parquet root tìm thấy:")
    for i, candidate in enumerate(candidates):
        print(f" [{i}] {candidate}")
    if len(candidates) > 1:
        print("Dùng candidate đầu tiên; đặt PARQUET_ROOT nếu muốn chọn root khác.")
    return candidates[0]


PARQUET_ROOT_PATH = discover_parquet_root(PARQUET_ROOT)
with (PARQUET_ROOT_PATH / "catalog.json").open("r", encoding="utf-8") as f:
    RAW_CATALOG = json.load(f)


def logical_table_dir(logical_path: str) -> Path:
    rel = Path(logical_path)
    clean_parts = [part.replace("-", "_") for part in rel.with_suffix("").parts]
    path = PARQUET_ROOT_PATH / "tables"
    for part in clean_parts:
        path = path / part
    return path


def table_exists(logical_path: str) -> bool:
    return any(logical_table_dir(logical_path).glob("*.parquet"))


def table_glob(logical_path: str) -> str:
    table_dir = logical_table_dir(logical_path)
    files = list(table_dir.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"Thiếu bảng {logical_path}: {table_dir}")
    return str(table_dir / "*.parquet")


CORE_TABLES = [
    "entities/course.json",
    "entities/problem.json",
    "entities/video.json",
    "entities/user.json",
    "relations/exercise-problem.txt",
    "relations/video_id-ccid.txt",
]
missing_core = [name for name in CORE_TABLES if not table_exists(name)]
if missing_core:
    raise FileNotFoundError("Thiếu các bảng bắt buộc: " + ", ".join(missing_core))

print("PARQUET_ROOT =", PARQUET_ROOT_PATH)
for name in CORE_TABLES:
    print(" OK", name, "->", table_glob(name))


In [ ]:
def total_memory_bytes() -> int:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (ValueError, OSError, AttributeError):
        return 30 * 1024**3


total_mem = total_memory_bytes()
memory_limit_gib = max(4, int(total_mem * MEMORY_FRACTION / 1024**3))
scratch_base = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
DUCKDB_TEMP = scratch_base / "moocubex_dimensions_duckdb"
DUCKDB_TEMP.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{memory_limit_gib}GB'")
con.execute(f"SET temp_directory='{str(DUCKDB_TEMP).replace(chr(39), chr(39) * 2)}'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET enable_progress_bar=true")

print(f"DuckDB memory_limit = {memory_limit_gib} GiB")
print("DuckDB temp_directory =", DUCKDB_TEMP)
disk_check_path = Path("/kaggle/working") if Path("/kaggle/working").exists() else OUTPUT_ROOT.parent
print("Free output filesystem =", f"{shutil.disk_usage(disk_check_path).free / 1024**3:.2f} GiB")


def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def normalize_id(expr: str, prefix: str) -> str:
    return f"""
        CASE
          WHEN {expr} IS NULL OR trim(CAST({expr} AS VARCHAR)) = '' THEN NULL
          WHEN starts_with(trim(CAST({expr} AS VARCHAR)), '{prefix}')
            THEN trim(CAST({expr} AS VARCHAR))
          ELSE '{prefix}' || trim(CAST({expr} AS VARCHAR))
        END
    """


SOURCE_VIEWS = {
    "src_course": "entities/course.json",
    "src_problem": "entities/problem.json",
    "src_video": "entities/video.json",
    "src_user": "entities/user.json",
    "src_exercise_problem": "relations/exercise-problem.txt",
    "src_video_ccid": "relations/video_id-ccid.txt",
}
OPTIONAL_SOURCE_VIEWS = {
    "src_comment": "entities/comment.json",
    "src_course_comment": "relations/course-comment.txt",
    "src_user_comment": "relations/user-comment.txt",
}

for view_name, logical_path in SOURCE_VIEWS.items():
    con.execute(
        f"CREATE OR REPLACE VIEW {view_name} AS "
        f"SELECT * FROM read_parquet({sql_string(table_glob(logical_path))}, union_by_name=true)"
    )

AVAILABLE_OPTIONAL: Dict[str, bool] = {}
for view_name, logical_path in OPTIONAL_SOURCE_VIEWS.items():
    available = table_exists(logical_path)
    AVAILABLE_OPTIONAL[view_name] = available
    if available:
        con.execute(
            f"CREATE OR REPLACE VIEW {view_name} AS "
            f"SELECT * FROM read_parquet({sql_string(table_glob(logical_path))}, union_by_name=true)"
        )
    print((" OK " if available else " -- "), logical_path)


## 2. Checkpoint writer và kiểm tra schema

Mỗi relation được ghi vào một thư mục riêng dưới `/kaggle/working/moocubex_model`. Nếu phiên chạy bị ngắt, chạy lại notebook sẽ bỏ qua các relation đã hoàn thành, trừ khi đặt `FORCE_REBUILD=True`.


In [ ]:
MANIFEST_PATH = OUTPUT_ROOT / "build_manifest.json"


def load_manifest() -> Dict[str, dict]:
    if not MANIFEST_PATH.exists():
        return {}
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        rows = json.load(f)
    return {row["name"]: row for row in rows}


def save_manifest() -> None:
    rows = sorted(build_manifest.values(), key=lambda row: row["name"])
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    tmp.replace(MANIFEST_PATH)


def output_path(relative_path: str) -> Path:
    return OUTPUT_ROOT / relative_path


def parquet_glob(path: Path) -> str:
    return str(path / "*.parquet")


def register_output(view_name: str, out_dir: Path) -> None:
    con.execute(
        f"CREATE OR REPLACE VIEW {view_name} AS "
        f"SELECT * FROM read_parquet({sql_string(parquet_glob(out_dir))}, union_by_name=true)"
    )


def remove_scoped_output(out_dir: Path) -> None:
    root = OUTPUT_ROOT.resolve()
    target = out_dir.resolve()
    if root not in target.parents:
        raise RuntimeError(f"Từ chối xóa path ngoài OUTPUT_ROOT: {target}")
    if out_dir.exists():
        shutil.rmtree(out_dir)


build_manifest = load_manifest()


def write_relation(name: str, relative_path: str, query: str) -> dict:
    out_dir = output_path(relative_path)
    out_file = out_dir / "part-00000.parquet"
    previous = build_manifest.get(name)

    if (
        not FORCE_REBUILD
        and previous
        and previous.get("status") == "success"
        and any(out_dir.glob("*.parquet"))
    ):
        register_output(name, out_dir)
        print(f"SKIP {name}: rows={previous.get('row_count', 0):,}")
        return previous

    remove_scoped_output(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    record = {
        "name": name,
        "relative_path": relative_path,
        "status": "running",
        "row_count": None,
        "output_bytes": 0,
        "elapsed_seconds": None,
        "error": None,
    }
    build_manifest[name] = record
    save_manifest()

    print("BUILD", name)
    try:
        con.execute(
            f"""
            COPY ({query})
            TO {sql_string(str(out_file))}
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL {ZSTD_LEVEL},
             ROW_GROUP_SIZE {ROW_GROUP_SIZE})
            """
        )
        register_output(name, out_dir)
        row_count = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
        output_bytes = sum(path.stat().st_size for path in out_dir.glob("*.parquet"))
        record.update({
            "status": "success",
            "row_count": int(row_count),
            "output_bytes": int(output_bytes),
            "elapsed_seconds": round(time.time() - started, 3),
        })
        print(
            f" OK rows={row_count:,} size={output_bytes / 1024**2:.1f} MiB "
            f"time={record['elapsed_seconds']:.1f}s"
        )
    except Exception as exc:
        record.update({
            "status": "error",
            "elapsed_seconds": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
        })
        print("ERROR", name, record["error"])
        raise
    finally:
        build_manifest[name] = record
        save_manifest()
    return record


REQUIRED_COLUMNS = {
    "src_course": {"id", "name", "resource"},
    "src_problem": {"problem_id", "exercise_id", "score"},
    "src_video": {"ccid", "end"},
    "src_user": {"id", "course_order", "enroll_time"},
    "src_exercise_problem": {"exercise_id", "problem_id"},
    "src_video_ccid": {"video_id", "ccid"},
}
for view_name, expected in REQUIRED_COLUMNS.items():
    actual = {row[0] for row in con.execute(f"DESCRIBE {view_name}").fetchall()}
    missing = expected - actual
    if missing:
        raise ValueError(f"{view_name} thiếu cột: {sorted(missing)}; hiện có: {sorted(actual)}")
    print("SCHEMA OK", view_name)


## 3. Dimensions course, problem, video và resource


In [ ]:
course_id = normalize_id("c.id", "C_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_course AS
SELECT
    {course_id} AS course_id,
    nullif(trim(c.name), '') AS course_name,
    c.field AS field,
    c.prerequisites AS prerequisites,
    c.about AS about,
    COALESCE(array_length(c.resource), 0) AS n_resource_rows
FROM src_course c
""")

write_relation(
    "quarantine_course_duplicate",
    "quality/quarantine_course_duplicate",
    """
    SELECT course_id, count(*) AS source_row_count,
           min(course_name) AS example_course_name
    FROM work_course
    WHERE course_id IS NULL
    GROUP BY course_id
    HAVING count(*) > 0
    UNION ALL
    SELECT course_id, count(*) AS source_row_count,
           min(course_name) AS example_course_name
    FROM work_course
    WHERE course_id IS NOT NULL
    GROUP BY course_id
    HAVING count(*) > 1
    """,
)

write_relation(
    "dim_course",
    "processed/dim_course",
    """
    WITH ranked AS (
      SELECT *,
             count(*) OVER (PARTITION BY course_id) AS source_row_count,
             row_number() OVER (
               PARTITION BY course_id
               ORDER BY n_resource_rows DESC, course_name NULLS LAST
             ) AS rn
      FROM work_course
      WHERE course_id IS NOT NULL
    )
    SELECT course_id, course_name, field, prerequisites, about,
           n_resource_rows, source_row_count
    FROM ranked
    WHERE rn = 1
    """,
)

problem_id = normalize_id("p.problem_id", "Pm_")
exercise_id = normalize_id("p.exercise_id", "Ex_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_problem AS
SELECT
    {problem_id} AS problem_id,
    {exercise_id} AS exercise_id,
    p.score AS problem_score_raw,
    p.type AS problem_type,
    p.typetext AS problem_type_text,
    p.language AS language
FROM src_problem p
""")

write_relation(
    "dim_problem",
    "processed/dim_problem",
    """
    SELECT
        problem_id,
        min(exercise_id) AS catalog_exercise_id,
        count(DISTINCT exercise_id) FILTER (WHERE exercise_id IS NOT NULL) AS n_catalog_exercises,
        max(problem_score_raw) FILTER (WHERE problem_score_raw >= 0) AS problem_max_score,
        CASE
          WHEN count(problem_score_raw) FILTER (WHERE problem_score_raw >= 0) = 0 THEN 'missing_or_negative'
          WHEN count(DISTINCT problem_score_raw) FILTER (WHERE problem_score_raw >= 0) > 1 THEN 'conflicting_valid_scores'
          ELSE 'valid'
        END AS score_catalog_status,
        min(problem_type) AS problem_type,
        min(problem_type_text) AS problem_type_text,
        min(language) AS language,
        count(*) AS source_row_count
    FROM work_problem
    WHERE problem_id IS NOT NULL
    GROUP BY problem_id
    """,
)

write_relation(
    "dim_video",
    "processed/dim_video",
    """
    WITH base AS (
      SELECT
          nullif(trim(ccid), '') AS ccid,
          nullif(trim(name), '') AS video_name,
          CASE
            WHEN "end" IS NULL OR COALESCE(array_length("end"), 0) = 0 THEN NULL
            ELSE list_max("end")
          END AS caption_end_seconds
      FROM src_video
    )
    SELECT
        ccid,
        min(video_name) AS video_name,
        max(caption_end_seconds) FILTER (WHERE caption_end_seconds > 0) AS available_video_seconds_proxy,
        CASE
          WHEN count(caption_end_seconds) FILTER (WHERE caption_end_seconds > 0) = 0 THEN 'missing'
          WHEN count(DISTINCT caption_end_seconds) FILTER (WHERE caption_end_seconds > 0) > 1 THEN 'conflicting_caption_end'
          ELSE 'caption_end_proxy'
        END AS duration_status,
        count(*) AS source_row_count
    FROM base
    WHERE ccid IS NOT NULL
    GROUP BY ccid
    """,
)


In [ ]:
resource_course_id = normalize_id("c.id", "C_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_lo_resource_raw AS
WITH exploded AS (
  SELECT
      {resource_course_id} AS course_id,
      nullif(trim(CAST(r.resource_id AS VARCHAR)), '') AS resource_id_raw,
      nullif(trim(CAST(r.chapter AS VARCHAR)), '') AS chapter_raw,
      lower(nullif(trim(CAST(r.resource_type AS VARCHAR)), '')) AS source_resource_type
  FROM src_course c
  CROSS JOIN UNNEST(c.resource) AS u(r)
)
SELECT
    course_id,
    CASE
      WHEN resource_id_raw IS NULL THEN NULL
      WHEN starts_with(resource_id_raw, 'V_') OR starts_with(resource_id_raw, 'Ex_') THEN resource_id_raw
      WHEN source_resource_type LIKE '%video%' THEN 'V_' || resource_id_raw
      WHEN source_resource_type LIKE '%exercise%' THEN 'Ex_' || resource_id_raw
      ELSE resource_id_raw
    END AS resource_id,
    chapter_raw,
    try_cast(regexp_extract(chapter_raw, '^([0-9]+)', 1) AS INTEGER) AS chapter_major,
    CASE
      WHEN starts_with(resource_id_raw, 'V_') THEN 'video'
      WHEN starts_with(resource_id_raw, 'Ex_') THEN 'exercise'
      WHEN source_resource_type LIKE '%video%' THEN 'video'
      WHEN source_resource_type LIKE '%exercise%' THEN 'exercise'
      ELSE 'unknown'
    END AS resource_type,
    source_resource_type
FROM exploded
""")

write_relation(
    "quarantine_lo_resource",
    "quality/quarantine_lo_resource",
    """
    SELECT *,
           concat_ws(';',
             CASE WHEN course_id IS NULL THEN 'missing_course_id' END,
             CASE WHEN resource_id IS NULL THEN 'missing_resource_id' END,
             CASE WHEN chapter_major IS NULL THEN 'invalid_chapter' END,
             CASE WHEN resource_type = 'unknown' THEN 'unknown_resource_type' END
           ) AS quarantine_reason
    FROM work_lo_resource_raw
    WHERE course_id IS NULL
       OR resource_id IS NULL
       OR chapter_major IS NULL
       OR resource_type = 'unknown'
    """,
)

write_relation(
    "dim_lo_resource",
    "processed/dim_lo_resource",
    """
    WITH deduplicated AS (
      SELECT
          course_id, chapter_major,
          course_id || ':CH' || CAST(chapter_major AS VARCHAR) AS lo_key,
          resource_id, resource_type,
          min(chapter_raw) AS chapter_raw_example,
          min(source_resource_type) AS source_resource_type,
          count(*) AS duplicate_count_within_lo
      FROM work_lo_resource_raw
      WHERE course_id IS NOT NULL
        AND resource_id IS NOT NULL
        AND chapter_major IS NOT NULL
        AND resource_type IN ('video', 'exercise')
      GROUP BY course_id, chapter_major, resource_id, resource_type
    ), resource_counts AS (
      SELECT
          resource_id,
          count(DISTINCT course_id) AS resource_course_count,
          count(DISTINCT lo_key) AS resource_lo_count,
          count(DISTINCT resource_type) AS resource_type_count
      FROM deduplicated
      GROUP BY resource_id
    )
    SELECT
        d.*,
        rc.resource_course_count, rc.resource_lo_count,
        (rc.resource_course_count > 1 OR rc.resource_lo_count > 1 OR rc.resource_type_count > 1)
          AS is_ambiguous_resource
    FROM deduplicated d
    JOIN resource_counts rc USING (resource_id)
    """,
)

write_relation(
    "dim_resource",
    "processed/dim_resource",
    """
    SELECT
        resource_id,
        min(resource_type) AS resource_type,
        count(DISTINCT resource_type) AS n_resource_types,
        count(DISTINCT course_id) AS n_courses,
        count(DISTINCT lo_key) AS n_los,
        max(is_ambiguous_resource) AS is_ambiguous_resource
    FROM dim_lo_resource
    GROUP BY resource_id
    """,
)


## 4. Bridges video, exercise–problem và LO–problem


In [ ]:
video_id = normalize_id("v.video_id", "V_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_video_ccid_pair AS
SELECT {video_id} AS video_id, nullif(trim(v.ccid), '') AS ccid, count(*) AS source_row_count
FROM src_video_ccid v
WHERE {video_id} IS NOT NULL AND nullif(trim(v.ccid), '') IS NOT NULL
GROUP BY 1, 2
""")

write_relation(
    "bridge_video_ccid",
    "processed/bridge_video_ccid",
    """
    WITH video_counts AS (
      SELECT video_id, count(*) AS n_ccids_per_video
      FROM work_video_ccid_pair GROUP BY video_id
    ), ccid_counts AS (
      SELECT ccid, count(*) AS n_video_ids_per_ccid
      FROM work_video_ccid_pair GROUP BY ccid
    )
    SELECT p.*, vc.n_ccids_per_video, cc.n_video_ids_per_ccid,
           (vc.n_ccids_per_video <> 1) AS is_ambiguous_video_id,
           (cc.n_video_ids_per_ccid <> 1) AS is_shared_ccid
    FROM work_video_ccid_pair p
    JOIN video_counts vc USING (video_id)
    JOIN ccid_counts cc USING (ccid)
    """,
)

rel_exercise_id = normalize_id("e.exercise_id", "Ex_")
rel_problem_id = normalize_id("e.problem_id", "Pm_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_exercise_problem_pair AS
WITH relation_pairs AS (
  SELECT {rel_exercise_id} AS exercise_id, {rel_problem_id} AS problem_id, 'relation' AS source
  FROM src_exercise_problem e
), catalog_pairs AS (
  SELECT exercise_id, problem_id, 'problem_catalog' AS source
  FROM work_problem
  WHERE exercise_id IS NOT NULL AND problem_id IS NOT NULL
), combined AS (
  SELECT * FROM relation_pairs
  UNION ALL
  SELECT * FROM catalog_pairs
)
SELECT
    exercise_id, problem_id,
    CASE
      WHEN count(DISTINCT source) = 2 THEN 'both'
      ELSE min(source)
    END AS mapping_source,
    count(*) AS source_row_count
FROM combined
WHERE exercise_id IS NOT NULL AND problem_id IS NOT NULL
GROUP BY exercise_id, problem_id
""")

write_relation(
    "bridge_exercise_problem",
    "processed/bridge_exercise_problem",
    "SELECT * FROM work_exercise_problem_pair",
)

write_relation(
    "quarantine_exercise_problem",
    "quality/quarantine_exercise_problem",
    """
    WITH lo_match AS (
      SELECT resource_id AS exercise_id, count(*) AS n_lo_matches
      FROM dim_lo_resource
      WHERE resource_type = 'exercise'
      GROUP BY resource_id
    )
    SELECT
        b.*, COALESCE(l.n_lo_matches, 0) AS n_lo_matches,
        (p.problem_id IS NOT NULL) AS problem_exists,
        concat_ws(';',
          CASE WHEN COALESCE(l.n_lo_matches, 0) = 0 THEN 'exercise_not_assigned_to_lo' END,
          CASE WHEN p.problem_id IS NULL THEN 'problem_not_in_catalog' END
        ) AS quarantine_reason
    FROM bridge_exercise_problem b
    LEFT JOIN lo_match l USING (exercise_id)
    LEFT JOIN dim_problem p USING (problem_id)
    WHERE COALESCE(l.n_lo_matches, 0) = 0 OR p.problem_id IS NULL
    """,
)

write_relation(
    "bridge_lo_problem",
    "processed/bridge_lo_problem",
    """
    WITH assigned AS (
      SELECT
          l.course_id, l.chapter_major, l.lo_key,
          b.exercise_id, b.problem_id, b.mapping_source,
          p.problem_max_score, p.score_catalog_status,
          l.is_ambiguous_resource
      FROM dim_lo_resource l
      JOIN bridge_exercise_problem b ON l.resource_id = b.exercise_id
      LEFT JOIN dim_problem p USING (problem_id)
      WHERE l.resource_type = 'exercise'
    ), counts AS (
      SELECT
          problem_id,
          count(DISTINCT course_id) AS problem_course_count,
          count(DISTINCT lo_key) AS problem_lo_count
      FROM assigned
      GROUP BY problem_id
    )
    SELECT
        a.*, c.problem_course_count, c.problem_lo_count,
        (c.problem_course_count > 1 OR c.problem_lo_count > 1) AS is_ambiguous_problem_assignment
    FROM assigned a
    JOIN counts c USING (problem_id)
    """,
)


## 5. `dim_lo_catalog` và enrollment bridge

`available_video_seconds_proxy` là `max(caption.end)` sau khi nối `video_id → ccid`; đây không phải duration thật. Mapping video mơ hồ không được đưa vào tổng duration.


In [ ]:
write_relation(
    "dim_lo_catalog",
    "processed/dim_lo_catalog",
    """
    WITH lo_base AS (
      SELECT
          course_id, chapter_major, lo_key,
          count(DISTINCT resource_id) FILTER (WHERE resource_type = 'video') AS n_videos_available,
          count(DISTINCT resource_id) FILTER (WHERE resource_type = 'exercise') AS n_exercises_available,
          count(DISTINCT resource_id) FILTER (WHERE is_ambiguous_resource) AS n_ambiguous_resources
      FROM dim_lo_resource
      GROUP BY course_id, chapter_major, lo_key
    ), problem_per_lo AS (
      SELECT
          course_id, lo_key, problem_id,
          max(problem_max_score) AS problem_max_score,
          max(score_catalog_status) AS score_catalog_status
      FROM bridge_lo_problem
      GROUP BY course_id, lo_key, problem_id
    ), problem_stats AS (
      SELECT
          course_id, lo_key, count(*) AS n_problems_available,
          sum(problem_max_score) FILTER (WHERE problem_max_score >= 0) AS available_max_score_sum,
          count(*) FILTER (WHERE problem_max_score IS NULL OR problem_max_score < 0) AS n_problems_missing_score
      FROM problem_per_lo
      GROUP BY course_id, lo_key
    ), unique_video_map AS (
      SELECT video_id, min(ccid) AS ccid
      FROM bridge_video_ccid
      WHERE n_ccids_per_video = 1
      GROUP BY video_id
    ), video_per_lo AS (
      SELECT
          l.course_id, l.lo_key, l.resource_id AS video_id,
          v.available_video_seconds_proxy
      FROM dim_lo_resource l
      LEFT JOIN unique_video_map m ON l.resource_id = m.video_id
      LEFT JOIN dim_video v ON m.ccid = v.ccid
      WHERE l.resource_type = 'video'
    ), video_stats AS (
      SELECT
          course_id, lo_key,
          count(*) FILTER (WHERE available_video_seconds_proxy > 0) AS n_videos_with_duration,
          count(*) FILTER (WHERE available_video_seconds_proxy IS NULL OR available_video_seconds_proxy <= 0)
            AS n_videos_missing_duration,
          sum(available_video_seconds_proxy) FILTER (WHERE available_video_seconds_proxy > 0)
            AS available_video_seconds_proxy
      FROM video_per_lo
      GROUP BY course_id, lo_key
    )
    SELECT
        b.course_id, c.course_name, b.chapter_major, b.lo_key,
        b.n_videos_available, b.n_exercises_available,
        COALESCE(p.n_problems_available, 0) AS n_problems_available,
        p.available_max_score_sum,
        COALESCE(p.n_problems_missing_score, 0) AS n_problems_missing_score,
        v.available_video_seconds_proxy,
        COALESCE(v.n_videos_with_duration, 0) AS n_videos_with_duration,
        COALESCE(v.n_videos_missing_duration, b.n_videos_available) AS n_videos_missing_duration,
        CASE WHEN COALESCE(v.n_videos_with_duration, 0) > 0
             THEN 'caption_end_proxy' ELSE 'unavailable' END AS duration_source,
        b.n_ambiguous_resources
    FROM lo_base b
    LEFT JOIN dim_course c USING (course_id)
    LEFT JOIN problem_stats p USING (course_id, lo_key)
    LEFT JOIN video_stats v USING (course_id, lo_key)
    """,
)

user_id = normalize_id("u.id", "U_")
con.execute(f"""
CREATE OR REPLACE TEMP VIEW work_user_enrollment AS
SELECT
    {user_id} AS user_id,
    u.course_order, u.enroll_time,
    COALESCE(array_length(u.course_order), 0) AS n_course_order,
    COALESCE(array_length(u.enroll_time), 0) AS n_enroll_time
FROM src_user u
""")

write_relation(
    "quarantine_user_enrollment_length",
    "quality/quarantine_user_enrollment_length",
    """
    SELECT user_id, n_course_order, n_enroll_time,
           'array_length_mismatch' AS quarantine_reason
    FROM work_user_enrollment
    WHERE n_course_order <> n_enroll_time OR user_id IS NULL
    """,
)

enrolled_course_id = normalize_id("list_extract(u.course_order, idx.array_index)", "C_")
write_relation(
    "bridge_user_course_enrollment",
    "processed/bridge_user_course_enrollment",
    f"""
    WITH eligible AS (
      SELECT * FROM work_user_enrollment
      WHERE user_id IS NOT NULL AND n_course_order = n_enroll_time
    ), expanded AS (
      SELECT
          u.user_id, idx.array_index,
          {enrolled_course_id} AS course_id,
          list_extract(u.enroll_time, idx.array_index) AS enroll_time_raw
      FROM eligible u
      CROSS JOIN UNNEST(range(1, u.n_course_order + 1)) AS idx(array_index)
    ), typed AS (
      SELECT
          *, try_cast(enroll_time_raw AS TIMESTAMP) AS enroll_time
      FROM expanded
    )
    SELECT
        t.user_id, t.course_id, t.enroll_time, t.enroll_time_raw, t.array_index,
        count(*) OVER (PARTITION BY t.user_id, t.course_id) AS user_course_enrollment_count,
        (c.course_id IS NOT NULL) AS course_exists_in_catalog,
        CASE
          WHEN t.enroll_time_raw IS NULL OR trim(t.enroll_time_raw) = '' THEN 'missing'
          WHEN t.enroll_time IS NULL THEN 'parse_error'
          ELSE 'valid'
        END AS enroll_time_status
    FROM typed t
    LEFT JOIN dim_course c USING (course_id)
    WHERE t.course_id IS NOT NULL
    """,
)


## 6. Comment bridges (tùy nguồn)

Comment chỉ được gán xuống LO khi `resource_id` nối duy nhất tới một LO. Nếu chỉ có `course-comment`, notebook giữ assignment ở cấp course; không suy diễn LO.


In [ ]:
if AVAILABLE_OPTIONAL.get("src_course_comment"):
    cc_course_id = normalize_id("x.course_id", "C_")
    cc_comment_id = normalize_id("x.comment_id", "Cm_")
    write_relation(
        "bridge_course_comment",
        "processed/bridge_course_comment",
        f"""
        WITH pairs AS (
          SELECT {cc_course_id} AS course_id, {cc_comment_id} AS comment_id, count(*) AS source_row_count
          FROM src_course_comment x
          GROUP BY 1, 2
        )
        SELECT p.*, (c.course_id IS NOT NULL) AS course_exists_in_catalog
        FROM pairs p LEFT JOIN dim_course c USING (course_id)
        WHERE p.course_id IS NOT NULL AND p.comment_id IS NOT NULL
        """,
    )
else:
    print("SKIP bridge_course_comment: thiếu relations/course-comment.txt")

if AVAILABLE_OPTIONAL.get("src_user_comment"):
    uc_user_id = normalize_id("x.user_id", "U_")
    uc_comment_id = normalize_id("x.comment_id", "Cm_")
    write_relation(
        "bridge_user_comment",
        "processed/bridge_user_comment",
        f"""
        SELECT {uc_user_id} AS user_id, {uc_comment_id} AS comment_id, count(*) AS source_row_count
        FROM src_user_comment x
        WHERE {uc_user_id} IS NOT NULL AND {uc_comment_id} IS NOT NULL
        GROUP BY 1, 2
        """,
    )
else:
    print("SKIP bridge_user_comment: thiếu relations/user-comment.txt")

if AVAILABLE_OPTIONAL.get("src_comment"):
    comment_id = normalize_id("cm.id", "Cm_")
    con.execute("""
        CREATE OR REPLACE TEMP VIEW work_resource_assignment AS
        SELECT
            resource_id,
            count(DISTINCT course_id) AS n_course_matches,
            count(DISTINCT lo_key) AS n_lo_matches,
            min(course_id) AS course_id_candidate,
            min(lo_key) AS lo_key_candidate,
            min(resource_type) AS resource_type
        FROM dim_lo_resource
        GROUP BY resource_id
    """)
    write_relation(
        "bridge_comment_resource",
        "processed/bridge_comment_resource",
        f"""
        SELECT
            {comment_id} AS comment_id,
            nullif(trim(cm.resource_id), '') AS resource_id,
            CASE WHEN a.n_course_matches = 1 THEN a.course_id_candidate END AS resolved_course_id,
            CASE WHEN a.n_lo_matches = 1 THEN a.lo_key_candidate END AS resolved_lo_key,
            a.resource_type,
            COALESCE(a.n_course_matches, 0) AS n_course_matches,
            COALESCE(a.n_lo_matches, 0) AS n_lo_matches,
            CASE
              WHEN cm.resource_id IS NULL OR trim(cm.resource_id) = '' THEN 'missing_resource_id'
              WHEN a.resource_id IS NULL THEN 'resource_not_in_catalog'
              WHEN a.n_lo_matches = 1 THEN 'resolved_unique_lo'
              ELSE 'ambiguous_resource'
            END AS assignment_status
        FROM src_comment cm
        LEFT JOIN work_resource_assignment a
          ON nullif(trim(cm.resource_id), '') = a.resource_id
        WHERE {comment_id} IS NOT NULL
        """,
    )
else:
    print("SKIP bridge_comment_resource: thiếu entities/comment.json")


## 7. Quality report, invariant tests và catalog output


In [ ]:
coverage_queries = [
    """SELECT 'exercise_problem_to_problem_catalog' AS metric,
              count(*) AS denominator,
              count(*) FILTER (WHERE p.problem_id IS NOT NULL) AS numerator
       FROM bridge_exercise_problem b LEFT JOIN dim_problem p USING (problem_id)""",
    """SELECT 'exercise_problem_to_any_lo' AS metric,
              count(*) AS denominator,
              count(*) FILTER (WHERE l.resource_id IS NOT NULL) AS numerator
       FROM bridge_exercise_problem b
       LEFT JOIN (SELECT DISTINCT resource_id FROM dim_lo_resource WHERE resource_type='exercise') l
         ON b.exercise_id = l.resource_id""",
    """SELECT 'lo_video_to_unique_ccid' AS metric,
              count(*) AS denominator,
              count(*) FILTER (WHERE v.video_id IS NOT NULL AND v.n_ccids_per_video = 1) AS numerator
       FROM dim_lo_resource l LEFT JOIN bridge_video_ccid v ON l.resource_id = v.video_id
       WHERE l.resource_type='video'""",
    """SELECT 'enrollment_to_course_catalog' AS metric,
              count(*) AS denominator,
              count(*) FILTER (WHERE course_exists_in_catalog) AS numerator
       FROM bridge_user_course_enrollment""",
]
if "bridge_comment_resource" in build_manifest and build_manifest["bridge_comment_resource"].get("status") == "success":
    coverage_queries.append(
        """SELECT 'comment_to_unique_lo' AS metric,
                  count(*) AS denominator,
                  count(*) FILTER (WHERE assignment_status='resolved_unique_lo') AS numerator
           FROM bridge_comment_resource"""
    )

coverage_union = " UNION ALL ".join(coverage_queries)
write_relation(
    "join_coverage_report",
    "quality/join_coverage_report",
    f"""
    WITH metrics AS ({coverage_union})
    SELECT metric, denominator, numerator,
           CASE WHEN denominator = 0 THEN NULL
                ELSE numerator::DOUBLE / denominator END AS coverage_rate
    FROM metrics
    """,
)

checks = {
    "dim_course_unique": "SELECT count(*) - count(DISTINCT course_id) FROM dim_course",
    "dim_problem_unique": "SELECT count(*) - count(DISTINCT problem_id) FROM dim_problem",
    "dim_video_unique": "SELECT count(*) - count(DISTINCT ccid) FROM dim_video",
    "dim_resource_unique": "SELECT count(*) - count(DISTINCT resource_id) FROM dim_resource",
    "dim_lo_unique": "SELECT count(*) - count(DISTINCT lo_key) FROM dim_lo_catalog",
    "invalid_chapter_in_dim_lo_resource": "SELECT count(*) FROM dim_lo_resource WHERE chapter_major IS NULL",
    "duplicate_lo_resource": """
        SELECT count(*) FROM (
          SELECT course_id, lo_key, resource_id, resource_type, count(*) AS n
          FROM dim_lo_resource GROUP BY 1,2,3,4 HAVING count(*) > 1
        )
    """,
    "duplicate_lo_problem": """
        SELECT count(*) FROM (
          SELECT course_id, lo_key, exercise_id, problem_id, count(*) AS n
          FROM bridge_lo_problem GROUP BY 1,2,3,4 HAVING count(*) > 1
        )
    """,
}
failed_checks = []
for check_name, query in checks.items():
    violations = int(con.execute(query).fetchone()[0])
    print(f"CHECK {check_name}: violations={violations:,}")
    if violations:
        failed_checks.append((check_name, violations))
if failed_checks:
    raise AssertionError(f"Invariant tests thất bại: {failed_checks}")

successful = [row for row in build_manifest.values() if row.get("status") == "success"]
audit_values = ",\n".join(
    "(" + ", ".join([
        sql_string(row["name"]),
        str(int(row.get("row_count") or 0)),
        str(int(row.get("output_bytes") or 0)),
        str(float(row.get("elapsed_seconds") or 0.0)),
    ]) + ")"
    for row in successful
)
write_relation(
    "catalog_audit",
    "quality/catalog_audit",
    f"""
    SELECT * FROM (VALUES {audit_values})
      AS t(table_name, row_count, output_bytes, elapsed_seconds)
    """,
)

save_manifest()
output_catalog = {
    "source_parquet_root": str(PARQUET_ROOT_PATH),
    "output_root": str(OUTPUT_ROOT),
    "created_with_duckdb": duckdb.__version__,
    "lo_definition": "big_chapter_first_numeric_component",
    "tables": {
        row["name"]: {
            "parquet_glob": str(OUTPUT_ROOT / row["relative_path"] / "*.parquet"),
            "row_count": row.get("row_count"),
            "output_bytes": row.get("output_bytes"),
        }
        for row in build_manifest.values()
        if row.get("status") == "success"
    },
}
with (OUTPUT_ROOT / "catalog.json").open("w", encoding="utf-8") as f:
    json.dump(output_catalog, f, ensure_ascii=False, indent=2)

(OUTPUT_ROOT / "_SUCCESS").write_text(
    "MOOCCubeX dimensions and bridges completed.\n", encoding="utf-8"
)
print("\nDONE")
print("Output:", OUTPUT_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Catalog:", OUTPUT_ROOT / 'catalog.json')


## Kết quả mong đợi

Sau khi notebook hoàn thành, publish toàn bộ `/kaggle/working/moocubex_model` thành một Kaggle Dataset để notebook normalize events đọc lại. Không chỉnh tay các giá trị quarantine; giữ chúng làm bằng chứng kiểm toán dữ liệu.

Các lưu ý quan trọng:

- resource xuất hiện ở nhiều course/LO vẫn được giữ nhưng gắn `is_ambiguous_resource`;
- enrollment có hai array lệch độ dài được quarantine thay vì zip sai;
- duration video chỉ là caption-end proxy;
- comment thiếu `resource_id` không được tự gán vào LO; có thể nối cấp course ở notebook social bằng `bridge_course_comment`.
